# relay_l2id_compare — four-arm head-to-head (new pipeline)

Ingests the `relay_l2id_compare` sweep (c4 + gen corpora) and compares the four relay designs using the **new registry-based kpi/ pipeline**.

**Source data** (already on disk, no re-simulation needed):
* `results_c4/` — 2100 sims (N=9 highway, 7 speeds × 30 seeds × 5 modes × 2 relays)
* `results_gen/` — 800 sims (N∈{9,16} × {highway,urban} × 2 op-points × 10 seeds × 5 modes)

**Inputs**: per-run KPI CSVs (one file per KPI: prr_type2, aoi, throughput, harq, ch_occupancy, storm, psr, coll) + summary CSV.

**Outputs**: the per-(arm, config) cell aggregate, the storm indicator, and the verdict table (H1/H2/H3 hypotheses).

Run order:
1. `python3 run_kpi_pipeline.py --runs-root results_c4 --out-dir /tmp/kpi_c4 --workers 8 --force` (and similarly for `results_gen`)
2. `python3 scratch_overlay/common/kpi/analyze_l2id_compare.py /tmp/kpi_c4 /tmp/kpi_c4_analysis` (and similarly for `results_gen`)
3. This notebook consumes the resulting `l2id_compare_*.csv` files.

In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve result dirs (override with L2ID_RESULTS env var).
_default_dirs = [
    "/Users/hadih/Documents/groupcast-adaptation-ns3/results_c4",
    "/Users/hadih/Documents/groupcast-adaptation-ns3/results_gen",
]
_env = os.environ.get("L2ID_RESULTS")
RESULTS_DIRS = _env.split(":") if _env else _default_dirs
RESULTS_DIRS = [d for d in RESULTS_DIRS if os.path.isdir(d)]
NOMINAL_BOUNDARY = 4
print("RESULTS_DIRS =", [os.path.abspath(d) for d in RESULTS_DIRS])

## 1. Load pre-computed aggregate tables

These were produced by `analyze_l2id_compare.py` from the per-KPI files emitted by `kpi_postprocess`. No need to re-walk the DBs.

In [ ]:
def load_aggregate(dir_):
    """Load the l2id_compare_aggregate.csv from one results dir."""
    p = Path(dir_) / "l2id_compare_aggregate.csv"
    if not p.exists():
        print(f"  missing: {p}")
        return pd.DataFrame()
    df = pd.read_csv(p)
    df["source"] = Path(dir_).name
    return df

def load_runs(dir_):
    p = Path(dir_) / "l2id_compare_runs.csv"
    if not p.exists():
        return pd.DataFrame()
    return pd.read_csv(p)

def load_verdict(dir_):
    p = Path(dir_) / "l2id_compare_verdict.csv"
    if not p.exists():
        return pd.DataFrame()
    return pd.read_csv(p)

def load_storm(dir_):
    p = Path(dir_) / "l2id_compare_storm_indicator.csv"
    if not p.exists():
        return pd.DataFrame()
    return pd.read_csv(p)

agg = pd.concat([load_aggregate(d) for d in RESULTS_DIRS], ignore_index=True, sort=False)
runs = pd.concat([load_runs(d) for d in RESULTS_DIRS], ignore_index=True, sort=False)
verdicts = pd.concat([load_verdict(d) for d in RESULTS_DIRS], ignore_index=True, sort=False)
storms = pd.concat([load_storm(d) for d in RESULTS_DIRS], ignore_index=True, sort=False)
print(f"agg={len(agg)}, runs={len(runs)}, verdicts={len(verdicts)}, storms={len(storms)}")
print("arms:", sorted(agg['arm'].dropna().unique()) if 'arm' in agg.columns else 'none')

## 2. Per-arm summaries

Aggregate over the configs to one row per arm class (fixed, broadcast, sameL2, adapt, none).

In [ ]:
# Collapse {fixed4, fixed6, fixed7, fixed11} → fixed
# Collapse {broadcast4, broadcast6, broadcast7, broadcast11} → broadcast
# Collapse {sameL24, sameL26, sameL27, sameL211} → sameL2
def collapse_arm(arm):
    a = str(arm).lower()
    for prefix, label in [("fixed", "fixed"), ("broadcast", "broadcast"),
                          ("samel", "sameL2"), ("none_adapt", "adapt"),
                          ("adapt", "adapt"), ("none", "none")]:
        if a.startswith(prefix):
            return label
    return arm

if not agg.empty and 'arm' in agg.columns:
    agg["arm_class"] = agg["arm"].apply(collapse_arm)
    per_arm = (agg.groupby("arm_class")
                  .agg({"storm_ratio": "mean",
                        "hop1_retx": "mean",
                        "ch_occ": "mean",
                        "coll_rate": "mean",
                        "harq_nack": "mean",
                        "prr_mean": "mean",
                        "prr_worst": "mean",
                        "prr_front": "mean",
                        "prr_tail": "mean",
                        "aoi_mean_ms": "mean",
                        "psr": "mean",
                        "d_prr_worst": "mean",
                        "d_prr_tail": "mean",
                        "d_prr_front": "mean",
                        "d_aoi_mean_ms": "mean",
                        "occ_ratio": "mean"})
                  .round(4))
    print(per_arm.to_string())
else:
    print("(no aggregate)")
    per_arm = pd.DataFrame()

## 3. Hypothesis plots

**Fig 1 (H1)** — feedback storm vs MCD. `sameL2` should climb with MCD; `fixed`/`broadcast` should sit flat near 1.0.

**Fig 2 (H2)** — tail-PRR retention. Δ tail-PRR vs none should be ≥ 0 for relay arms.

**Fig 3 (H3)** — total resource cost. Occupancy ratio (relay / none) — fixed costs ~50% more, broadcast is cheaper.

**Fig 4 (H4)** — RB-collision trade. Front-PRR degradation + leader↔relay RB-overlap rate.

In [ ]:
if not storms.empty:
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for arm in ["sameL2", "fixed", "broadcast", "adapt", "none"]:
        sub = storms[storms["arm"].apply(collapse_arm) == arm]
        if len(sub) == 0:
            continue
        g = sub.groupby("mcdMeters")["storm_ratio_hop1_mean"].agg(["mean", "std"])
        ax.errorbar(g.index, g["mean"], yerr=g["std"], marker="o", label=arm, capsize=3)
    ax.axhline(1.0, ls="--", c="grey", lw=1)
    ax.set_xlabel("MCD (m)")
    ax.set_ylabel("hop-1 retx ratio (arm / none)")
    ax.set_title("H1 — feedback storm vs MCD")
    ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("(no storm data)")

In [ ]:
if not agg.empty:
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for arm in ["sameL2", "fixed", "broadcast", "adapt"]:
        sub = agg[agg["arm"].apply(collapse_arm) == arm]
        if len(sub) == 0:
            continue
        g = sub.groupby("mcdMeters")["d_prr_tail"].agg(["mean", "std"])
        ax.errorbar(g.index, g["mean"], yerr=g["std"], marker="o", label=arm, capsize=3)
    ax.axhline(0, ls="--", c="grey", lw=1)
    ax.set_xlabel("MCD (m)")
    ax.set_ylabel("Δ tail-PRR vs none")
    ax.set_title("H2 — tail-PRR benefit vs MCD")
    ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

In [ ]:
if not agg.empty:
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for arm in ["sameL2", "fixed", "broadcast", "adapt"]:
        sub = agg[agg["arm"].apply(collapse_arm) == arm]
        if len(sub) == 0:
            continue
        g = sub.groupby("mcdMeters")["occ_ratio"].agg(["mean", "std"])
        ax.errorbar(g.index, g["mean"], yerr=g["std"], marker="o", label=arm, capsize=3)
    ax.axhline(1.0, ls="--", c="grey", lw=1)
    ax.set_xlabel("MCD (m)")
    ax.set_ylabel("channel occupancy ratio (arm / none)")
    ax.set_title("H3 — total resource cost vs MCD")
    ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

In [ ]:
if not agg.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for arm in ["sameL2", "fixed", "broadcast"]:
        sub = agg[agg["arm"].apply(collapse_arm) == arm]
        if len(sub) == 0:
            continue
        g = sub.groupby("mcdMeters")["d_prr_front"].agg(["mean", "std"])
        axes[0].errorbar(g.index, g["mean"], yerr=g["std"], marker="o", label=arm, capsize=3)
    axes[0].axhline(0, ls="--", c="grey", lw=1)
    axes[0].set_xlabel("MCD (m)")
    axes[0].set_ylabel("Δ front-PRR vs none")
    axes[0].set_title("H4a — front PRR (collision impact)")
    axes[0].legend(); axes[0].grid(alpha=.3)
    # H4b: collision rate by arm position
    cr = (agg[agg["arm_class"].isin(["sameL2", "fixed", "broadcast"])]
          .assign(lbl=lambda d: [f"{a.split('_')[0]}@{int(rn)}"
                                   for a, rn in zip(d["arm"], d["relayNode"])])
          .groupby("lbl")["coll_rate"].mean().sort_values())
    if cr.notna().any():
        axes[1].bar(cr.index, cr.values)
    axes[1].set_ylabel("leader↔relay RB-collision rate")
    axes[1].set_title("H4b — collision rate by arm position")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

In [ ]:
if not agg.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    lo = agg[agg["mcdMeters"] <= 150]
    for a in ["sameL2", "fixed", "broadcast"]:
        for rn, g in lo[lo["arm_class"] == a].groupby("relayNode"):
            x = g["occ_ratio"].mean(); y = g["d_prr_tail"].mean()
            if np.isfinite(x) and np.isfinite(y):
                ax.scatter(x, y, s=90)
                ax.annotate(f"{a}@{int(rn)}", (x, y),
                            textcoords="offset points", xytext=(6, 4), fontsize=9)
    ax.axhline(0, ls="--", c="grey", lw=1)
    ax.set_xlabel("channel occupancy ratio vs none  (cost →)")
    ax.set_ylabel("Δ tail-PRR vs none  (benefit ↑)")
    ax.set_title("Benefit vs cost (MCD ≤ 150 m)")
    ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 4. Verdict table

One row per arm class, averaged across the corpus. The hypotheses are:
* **H1_no_storm** — `storm_ratio_hiMCD ≤ 1.20` (relay retx pressure ≤ none's).
* **H2_tail_kept** — `Δ tail-PRR ≥ -0.005` (no measurable tail regression).
* **H3_cheaper_than_sameL2** — `occ_ratio < 1.5` (≤ 50% extra channel).

In [ ]:
if not verdicts.empty:
    verdicts["arm_class"] = verdicts["arm"].apply(collapse_arm)
    collapsed = (verdicts.groupby("arm_class")
                       .agg({"storm_ratio_hiMCD": "mean",
                             "d_tail_PRR": "mean",
                             "d_front_PRR": "mean",
                             "occ_ratio": "mean",
                             "coll_rate": "mean",
                             "d_AoI_ms": "mean"})
                       .round(4).reset_index())
    # Re-evaluate hypotheses
    collapsed["H1_no_storm"] = collapsed["storm_ratio_hiMCD"].apply(
        lambda x: bool(x is not None and not pd.isna(x) and x <= 1.20))
    collapsed["H2_tail_kept"] = collapsed["d_tail_PRR"].apply(
        lambda x: bool(x is not None and not pd.isna(x) and x >= -0.005))
    s_occ = collapsed.loc[collapsed["arm_class"] == "sameL2", "occ_ratio"]
    s_occ = float(s_occ.iloc[0]) if len(s_occ) > 0 else 1.5
    collapsed["H3_cheaper_than_sameL2"] = collapsed["occ_ratio"].apply(
        lambda x: bool(x is not None and not pd.isna(x) and x < s_occ))
    print(collapsed.to_string(index=False))
    collapsed.to_csv("l2id_compare_verdict_collapsed.csv", index=False)